# Monitoraggio della reputazione online di un'azienda

> **Azienda**: MachineInnovators Inc. — leader nello sviluppo di applicazioni di machine learning scalabili e pronte per la produzione

> **Problema**: monitorare manualmente il sentiment degli utenti sui social media e' inefficiente, soggetto a errori umani e troppo lento per intervenire in tempo su un calo di reputazione

> **Soluzione proposta**: automatizzare l'analisi del sentiment con un modello pre-addestrato e costruire attorno ad esso una pipeline MLOps reale di training, test, deploy e monitoraggio continuo

> **Modello richiesto**: [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)

> **Dataset pubblico**: [`tweet_eval`, task `sentiment`](https://huggingface.co/datasets/cardiffnlp/tweet_eval)

## Contesto

Il modo in cui un'azienda viene percepita sui social media influenza direttamente vendite, fiducia degli investitori e valore del brand. Il volume di menzioni (tweet, post, recensioni) che un'azienda di medie o grandi dimensioni riceve ogni giorno rende impossibile una lettura manuale sistematica: servono strumenti automatici capaci di leggere grandi quantita' di testo e restituire un giudizio sintetico e strutturato — questo e' il ruolo dell'NLP in questo progetto.

MachineInnovators Inc. vuole integrare metodologie **MLOps** per non fermarsi alla singola previsione, ma costruire un flusso completo che copra lo sviluppo, il testing, il deploy e il monitoraggio continuo del modello di analisi del sentiment. L'obiettivo di business e' abilitare l'azienda a migliorare e monitorare la propria reputazione sui social media in modo tempestivo — rilevando un peggioramento del sentiment mentre e' ancora gestibile, non dopo che e' gia' diventato un problema pubblico.

## Obiettivo del notebook

1. Caricare `tweet_eval/sentiment`, un dataset pubblico di tweet etichettati come `negative`/`neutral`/`positive`, e controllarne la qualita' (valori mancanti, duplicati, bilanciamento delle classi).
2. Caricare il modello pre-addestrato richiesto (`cardiffnlp/twitter-roberta-base-sentiment-latest`) ed eseguire l'inferenza su un campione del test set.
3. Valutare le prestazioni con metriche adatte a un problema multiclasse potenzialmente sbilanciato: accuracy, precision/recall/F1 macro e weighted, matrice di confusione, confidence.
4. Descrivere (non eseguire qui) la **pipeline CI/CD** automatizzata per training, test di integrazione e deploy, e il **sistema di monitoraggio continuo** — entrambi implementati come repository GitHub reale, non simulati nel notebook — con il link al repository pubblico richiesto dalla consegna.

## Metodologia

Il modello richiesto e' un Transformer (RoBERTa) gia' specializzato per la sentiment analysis su Twitter: viene usato **in inferenza diretta, senza fine-tuning**, e valutato sul benchmark pubblico `tweet_eval`. Il valore aggiunto del notebook non sta quindi nell'addestrare un classificatore da zero, ma nel valutare criticamente le prestazioni del modello gia' pronto.

I parametri di esecuzione (seed, batch size, numero di esempi valutati) sono centralizzati in un'unica `Config`: un solo punto da modificare invece di costanti sparse nel notebook.

Il training automatizzato, i test di integrazione, il deploy su HuggingFace e il monitoraggio continuo (Fase 2 e Fase 3 della consegna) **non vengono eseguiti in questo notebook**: sono implementati come pipeline reale nella repository GitHub (sezione 10), con codice che gira davvero tramite GitHub Actions — coerente con la consegna, che chiede una pipeline automatizzata e un sistema di monitoraggio, non una loro simulazione dentro un notebook.

## Struttura del notebook

0. Link al repository GitHub del progetto
1. Installazione librerie
2. Importazioni e configurazione centralizzata (`Config`)
3. Caricamento del dataset (`tweet_eval/sentiment`)
4. Controllo qualita' dei dati
5. Analisi esplorativa (distribuzione delle classi, lunghezza dei testi, esempi)
6. Caricamento del modello pre-addestrato
7. Funzioni modulari di inference e valutazione
8. Inference sul test set
9. Valutazione delle performance (metriche, matrice di confusione, confidence)
10. Pipeline CI/CD e monitoraggio continuo (descrizione della repository GitHub)
11. Conclusioni finali

Questo notebook e' pensato per essere eseguito su **Google Colab**.
5. Eseguire su Colab lo stesso script di retraining usato da GitHub Actions, con backbone congelato e controllo di regressione (sezione 9-bis).


## 0. Link repository GitHub

La consegna richiede che il notebook contenga il link al repository GitHub pubblico. Dopo aver creato il repository e caricato i file, sostituire il placeholder qui sotto.

In [ ]:
# Link al repository GitHub del progetto.
# Monorepo con tutti i progetti d'esame; questo link punta direttamente alla
# cartella con il codice di questo progetto (predictor, app, test, training,
# monitoraggio, pipeline CI/CD in .github/workflows/).
GITHUB_REPOSITORY_URL = "https://github.com/giuli-c/Folder-ML-Projects/tree/main/Monitoraggio%20della%20reputazione%20online%20di%20un%E2%80%99azienda/sentiment_reputation_mlops"

print(f"Repository GitHub progetto: {GITHUB_REPOSITORY_URL}")


## 1. Installazione librerie

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib seaborn pandas numpy gradio huggingface_hub

## 2. Importazioni e configurazione

In [ ]:
# ============================================================
# IMPORTAZIONI E CONFIGURAZIONE
# ============================================================

import os
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

sns.set_theme(style="whitegrid")

In [ ]:
# ============================================================
# CELLA — Config: parametri centralizzati del progetto
# ============================================================
@dataclass
class Config:
    seed: int = 42
    model_name: str = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    dataset_name: str = "cardiffnlp/tweet_eval"
    dataset_task: str = "sentiment"
    max_eval_samples: int = 1000
    batch_size: int = 32
    label_map: Dict[int, str] = None

    def __post_init__(self):
        if self.label_map is None:
            # tweet_eval/sentiment usa questa codifica ufficiale:
            # 0 = negative, 1 = neutral, 2 = positive.
            self.label_map = {0: "negative", 1: "neutral", 2: "positive"}

cfg = Config()

In [ ]:
# ============================================================
# Impostazione del Seed
# ============================================================
def set_seed(seed: int = 42) -> None:
    """Rende piu' riproducibili campionamento e risultati."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
device = 0 if torch.cuda.is_available() else -1
print("Device usato:", "GPU" if device == 0 else "CPU")

## 3. Caricamento del dataset

Il dataset scelto e' `cardiffnlp/tweet_eval` (subtask `sentiment`), una raccolta pubblica di tweet etichettati come negativi, neutri o positivi. E' coerente con il modello CardiffNLP richiesto, perche' entrambi lavorano sul linguaggio tipico di Twitter/social media — non a caso condividono anche lo stesso namespace su HuggingFace Hub.

In [ ]:
# ============================================================
# CARICAMENTO DATASET PUBBLICO
# ============================================================

raw_dataset = load_dataset(cfg.dataset_name, cfg.dataset_task)

print(raw_dataset)
print("\nSplit disponibili:", list(raw_dataset.keys()))

In [ ]:
train_df = raw_dataset["train"].to_pandas()
val_df = raw_dataset["validation"].to_pandas()
test_df = raw_dataset["test"].to_pandas()

for df in [train_df, val_df, test_df]:
    df["sentiment"] = df["label"].map(cfg.label_map)
    df["text_length"] = df["text"].str.len()
    df["word_count"] = df["text"].str.split().str.len()

print("Dimensioni train/validation/test:")
print(train_df.shape, val_df.shape, test_df.shape)

train_df.head()

## 4. Controllo qualita' dei dati

Prima di usare un modello e' utile controllare valori mancanti, duplicati e distribuzione delle etichette. 

In [ ]:
# ============================================================
# DATA QUALITY CHECK
# ============================================================

def data_quality_report(df: pd.DataFrame, name: str) -> None:
    print(f"--- {name.upper()} ---")
    print(f"Righe: {len(df):,}")
    print("Valori mancanti:")
    print(df.isnull().sum())
    print(f"Duplicati sul testo: {df['text'].duplicated().sum():,}")
    print("Distribuzione label:")
    print(df["sentiment"].value_counts().to_string())
    print()

data_quality_report(train_df, "train")
data_quality_report(val_df, "validation")
data_quality_report(test_df, "test")

## 5. Analisi esplorativa

Questa sezione serve a capire la forma dei dati prima della valutazione del modello: quante classi ci sono, quanto sono bilanciate e quanto sono lunghi i testi. Sono informazioni semplici, ma aiutano molto a interpretare i risultati successivi.

In [ ]:
# ============================================================
# GRAFICO 1: DISTRIBUZIONE DELLE CLASSI
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, df) in zip(axes, [("Train", train_df), ("Validation", val_df), ("Test", test_df)]):
    order = ["negative", "neutral", "positive"]
    counts = df["sentiment"].value_counts().reindex(order)
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette="Set2")
    ax.set_title(f"Distribuzione sentiment - {name}")
    ax.set_xlabel("Sentiment")
    ax.set_ylabel("Numero testi")
    for i, val in enumerate(counts.values):
        ax.text(i, val + max(counts.values) * 0.02, f"{val:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## Analisi della qualità e della distribuzione del dataset

### Controllo degli split

Il dataset è suddiviso in:

- **Train:** 45.615 esempi
- **Validation:** 2.000 esempi
- **Test:** 12.284 esempi

Il controllo preliminare non evidenzia **valori mancanti** nelle variabili analizzate (`text`, `label`, `sentiment`, `text_length`, `word_count`) in nessuno dei tre split.

Sono stati rilevati **29 testi duplicati nel Train**, mentre Validation e Test non presentano duplicati. Considerata la dimensione del training set, si tratta comunque di una quota molto ridotta.

### Distribuzione delle classi

| Split | Negative | Neutral | Positive |
|---|---:|---:|---:|
| **Train** | 7.093 (15,5%) | 20.673 (45,3%) | 17.849 (39,1%) |
| **Validation** | 312 (15,6%) | 869 (43,5%) | 819 (41,0%) |
| **Test** | 3.972 (32,3%) | 5.937 (48,3%) | 2.375 (19,3%) |

Guardando le percentuali, Train e Validation si assomigliano molto: in entrambi la classe più comune è `neutral`, poi `positive`, e `negative` è quella con meno esempi.

Il **Test set presenta invece una composizione sensibilmente diversa**. 

La classe `negative` passa da circa **15–16% a 32,3%**, quindi quasi raddoppia, mentre `positive` scende da circa **39–41% a 19,3%**, circa la metà. `Neutral` rimane invece la classe più rappresentata in tutti gli split.

→ È quindi presente un **distribution shift nelle proporzioni delle classi tra Train/Validation e Test**.

Questo aspetto dovrà essere considerato nell'interpretazione delle metriche finali, perché il modello verrà valutato su una distribuzione delle label diversa da quella osservata durante training e validation.

In [ ]:
# ============================================================
# GRAFICO 2: LUNGHEZZA DEI TESTI PER SENTIMENT
# ============================================================

plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="sentiment", y="word_count", order=["negative", "neutral", "positive"], palette="Set2")
plt.title("Distribuzione della lunghezza dei testi per sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Numero parole")
plt.tight_layout()
plt.show()

### Osservazioni sulla lunghezza dei testi

Dal grafico si può osservare che la **lunghezza dei testi è abbastanza simile nelle tre classi di sentiment**.

- I testi `negative` hanno una lunghezza mediana leggermente maggiore, circa **21 parole**.
- I testi `neutral` e `positive` hanno invece una mediana di circa **19 parole**.
- La maggior parte dei testi, per tutte le classi, si concentra indicativamente tra **16 e 24 parole**.
- Sono presenti alcuni **valori anomali (outlier)**, soprattutto testi molto brevi con meno di 5-6 parole e alcuni testi più lunghi, intorno alle 34-35 parole.

Nel complesso, non si notano grandi differenze nella lunghezza dei testi tra `negative`, `neutral` e `positive`.

→ La **lunghezza del testo non sembra quindi essere una caratteristica che distingue in modo evidente le tre classi**: il sentiment dovrà essere riconosciuto principalmente a partire dal contenuto delle frasi e non semplicemente dal numero di parole.

In [ ]:
# ============================================================
# GRAFICO 3: ESEMPI DI TESTI PER CLASSE
# ============================================================

for sentiment in ["negative", "neutral", "positive"]:
    print(f"\n--- Esempi classe {sentiment.upper()} ---")
    sample_texts = train_df[train_df["sentiment"] == sentiment].sample(3, random_state=cfg.seed)["text"].tolist()
    for idx, text in enumerate(sample_texts, start=1):
        print(f"{idx}. {text}")

### Osservazioni sugli esempi testuali

Osservando alcuni esempi casuali per ciascuna classe si nota che i testi hanno le caratteristiche tipiche di messaggi provenienti da **Twitter/X**: sono brevi, informali e possono contenere **hashtag, menzioni (`@user`), abbreviazioni, nomi propri e punteggiatura usata per enfatizzare il messaggio**.

Negli esempi `negative` sono presenti espressioni che comunicano chiaramente insoddisfazione o frustrazione, mentre nei `positive` compaiono messaggi con un tono più favorevole o entusiasta. I testi `neutral` risultano invece più descrittivi o informativi e non esprimono un'opinione particolarmente positiva o negativa.

Si nota inoltre che il sentiment non dipende necessariamente da singole parole, ma dal **significato complessivo della frase e dal contesto**.

→ Questi esempi mostrano quindi che la classificazione del sentiment richiede di comprendere il contenuto del testo, tenendo conto anche del linguaggio informale tipico dei social network.

## 6. Caricamento del modello pre-addestrato

Il modello richiesto e' una versione RoBERTa addestrata per sentiment analysis su Twitter. Usiamo una pipeline HuggingFace per tenere il codice leggibile e adatto a Colab.

In [ ]:
# ============================================================
# MODELLO HUGGINGFACE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name)

# Creazione una pipeline Hugging Face per la classificazione del sentiment.
# La pipeline si occupa automaticamente di:
# - tokenizzare il testo con il tokenizer scelto;
# - passare gli input al modello;
# - eseguire la previsione;
# - restituire la classe prevista con il relativo score.
#
# `task` indica a Hugging Face quale operazione vogliamo eseguire.
# Alcuni esempi:
# - task="text-classification"     -> classificazione generica di un testo
#   es. "This email is spam" -> spam
# - task="text-generation"         -> generazione di testo
#   es. "Once upon a time..." -> continua la frase
# - task="summarization"           -> riassunto di un testo
#   es. testo lungo -> breve riassunto
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=128,
)

print("Modello caricato:", cfg.model_name)
print("Etichette modello:", model.config.id2label)

## 7. Funzioni modulari di inference e valutazione

In [ ]:
# ============================================================
# FUNZIONI DI UTILITA'
# ============================================================

def normalize_model_label(label: str) -> str:
    """Converte eventuali label HuggingFace tipo LABEL_0 nei nomi sentiment."""
    label = label.lower()
    if label.startswith("label_"):
        label_id = int(label.replace("label_", ""))
        return cfg.label_map[label_id]
    return label

def predict_sentiment(texts: List[str], batch_size: int = 32, model_pipeline=None) -> Tuple[List[str], List[float]]:
    """
    Predice sentiment e confidence score per una lista di testi.
    Usa `sentiment_pipeline` (il modello originale) di default; passare
    `model_pipeline` per riusare la stessa logica di inferenza con un altro
    modello.
    """
    pipe = model_pipeline if model_pipeline is not None else sentiment_pipeline
    predictions = []
    scores = []

    for start in range(0, len(texts), batch_size):
        # prendo una porzione della lista texts
        # ["frase 0", "frase 1"]
        batch = texts[start:start + batch_size]
        outputs = pipe(batch, batch_size=batch_size)
        for out in outputs:
            if start in range(0, 10):
              print(out)
            predictions.append(normalize_model_label(out["label"]))
            scores.append(float(out["score"]))

    return predictions, scores

def evaluate_sentiment_model(df: pd.DataFrame, split_name: str) -> Dict[str, float]:
    """
    Valuta il modello su uno split e restituisce metriche principali.
    """
    y_true = df["sentiment"].tolist()
    y_pred = df["predicted_sentiment"].tolist()

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

def sample_for_test(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """
    Campiona lo split per rendere l'esecuzione sostenibile su Colab.
    """
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

## 8. Inference sul test set

Per mantenere il notebook veloce in Colab, la valutazione usa un campione del test set. In un ambiente di produzione useremmo l'intero test set o una suite di benchmark versionata.

In [ ]:
# ============================================================
# PREDIZIONE SUL TEST SET
# ============================================================

sample_test_df = sample_for_test(test_df, cfg.max_eval_samples, cfg.seed)

start_time = time.time()
preds, scores = predict_sentiment(sample_test_df["text"].tolist(), batch_size=cfg.batch_size)
elapsed = time.time() - start_time

sample_test_df["predicted_sentiment"] = preds
sample_test_df["confidence"] = scores

print(f"Esempi valutati: {len(sample_test_df):,}")
print(f"Tempo inference: {elapsed:.2f} secondi")
print(f"Tempo medio per testo: {elapsed / len(sample_test_df):.4f} secondi")

sample_test_df[["text", "sentiment", "predicted_sentiment", "confidence"]].head()

## 9. Valutazione delle performance

Usiamo piu' metriche per evitare una lettura troppo superficiale. L'accuracy e' intuitiva, ma con tre classi e possibile sbilanciamento e' importante guardare anche macro precision, macro recall e macro F1.

In [ ]:
# ============================================================
# METRICHE DI VALUTAZIONE
# ============================================================

metrics = evaluate_sentiment_model(sample_test_df, "test_sample")
metrics_df = pd.DataFrame([metrics]).set_index("split")
display(metrics_df.style.format("{:.4f}"))

print("Classification report:\n")
print(classification_report(
    sample_test_df["sentiment"],
    sample_test_df["predicted_sentiment"],
    labels=["negative", "neutral", "positive"],
    zero_division=0,
))

In [ ]:
# ============================================================
# GRAFICO 4: METRICHE PRINCIPALI
# ============================================================

metric_order = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]
plot_metrics = metrics_df.loc["test_sample", metric_order]

plt.figure(figsize=(10, 4))
sns.barplot(x=plot_metrics.index, y=plot_metrics.values, palette="viridis")
plt.ylim(0, 1)
plt.title("Metriche di performance sul campione di test")
plt.ylabel("Score")
plt.xlabel("Metrica")
plt.xticks(rotation=20, ha="right")
for i, val in enumerate(plot_metrics.values):
    plt.text(i, val + 0.02, f"{val:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

### Osservazione sulle metriche

Il modello raggiunge un'**accuracy del 70%** sul campione di test, con valori simili anche per le altre metriche principali:

- **Precision macro:** 0.699
- **Recall macro:** 0.710
- **F1-score macro:** 0.702
- **F1-score weighted:** 0.698

Le prestazioni risultano quindi **abbastanza bilanciate tra le classi**.

Analizzando i risultati per singola classe:

- **Negative** → è la classe riconosciuta meglio, con **F1-score = 0.73** e **recall = 0.79**.
- **Positive** → mostra prestazioni equilibrate, con **F1-score = 0.70**.
- **Neutral** → risulta la classe più difficile da identificare, con **recall = 0.63** e **F1-score = 0.67**.

Nel complesso, il modello mostra **prestazioni discrete e abbastanza uniformi**, ma presenta maggiore difficoltà nel riconoscimento dei testi **neutrali**.

In [ ]:
# ============================================================
# GRAFICO 5: MATRICE DI CONFUSIONE
# ============================================================

labels = ["negative", "neutral", "positive"]
cm = confusion_matrix(sample_test_df["sentiment"], sample_test_df["predicted_sentiment"], labels=labels)
cm_df = pd.DataFrame(cm, index=[f"Reale {l}" for l in labels], columns=[f"Pred {l}" for l in labels])

plt.figure(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matrice di confusione - sentiment")
plt.tight_layout()
plt.show()

### Osservazione sulla matrice di confusione

La matrice di confusione permette di osservare **quali classi vengono riconosciute correttamente e quali vengono confuse tra loro**.

- **Negative:** 259 esempi su 328 vengono classificati correttamente. L'errore principale è la classificazione come **neutral** (63 casi).
- **Neutral:** 293 esempi su 462 sono corretti. È la classe che genera più errori: 110 esempi vengono classificati come **negative** e 59 come **positive**.
- **Positive:** 148 esempi su 210 vengono classificati correttamente. Gli errori sono soprattutto verso la classe **neutral** (53 casi), mentre solo 9 vengono classificati come **negative**.

Nel complesso, il modello distingue abbastanza bene i sentimenti **negative** e **positive**, mentre mostra maggiore difficoltà con la classe **neutral**, che tende a sovrapporsi alle altre due.

È inoltre interessante notare che la confusione diretta tra **positive** e **negative** è molto bassa (6 e 9 casi): gli errori avvengono soprattutto passando attraverso la classe **neutral**.

In [ ]:
# ============================================================
# GRAFICO 6: CONFIDENCE DEL MODELLO
# ============================================================

plt.figure(figsize=(10, 5))
sns.histplot(data=sample_test_df, x="confidence", hue="predicted_sentiment", bins=25, kde=True, palette="Set2")
plt.title("Distribuzione della confidence per sentiment predetto")
plt.xlabel("Confidence del modello")
plt.ylabel("Numero testi")
plt.tight_layout()
plt.show()

### Osservazione sulla confidence del modello

Il grafico mostra **quanto il modello è sicuro delle proprie predizioni**, distinguendo i risultati in base al sentiment predetto.

Per leggere il grafico:
- sull'**asse X** è riportata la **confidence**, cioè il livello di sicurezza associato alla previsione: valori più vicini a **1** indicano una maggiore sicurezza;
- sull'**asse Y** è riportato il **numero di testi** che presentano un determinato intervallo di confidence;
- i diversi **colori** rappresentano le tre classi predette: `negative`, `neutral` e `positive`;
- le **linee** aiutano a visualizzare l'andamento generale della distribuzione della confidence per ciascuna classe.

Dal grafico emerge che le predizioni **positive** sono spesso associate a confidence molto elevate, concentrate soprattutto verso **0.90–1.00**. Anche per la classe **negative** il modello mostra generalmente una buona sicurezza, con numerose predizioni ad alta confidence.

La classe **neutral** presenta invece una distribuzione più ampia, con molti valori anche tra circa **0.50 e 0.80**. Questo indica che il modello tende ad essere **meno sicuro quando assegna un testo alla classe neutral**.

Il risultato è coerente con le analisi precedenti: la classe `neutral` è quella che presenta il **recall e l'F1-score più bassi** e, nella matrice di confusione, è anche quella maggiormente confusa con le altre classi.

> **Nota:** una confidence elevata indica che il modello è molto sicuro della propria previsione, ma **non garantisce che la previsione sia corretta**. 

### Dataset interno: dai post Mastodon alle etichette approvate

Questa versione usa come fonte principale `monitoring/review_queue.json`.
La raccolta avviene nel repository tramite `monitor.py`; la revisione reale si effettua modificando manualmente il JSON, come descritto nel README.
Compila `validated_label`, `review_status`, `reviewer` e `reviewed_at`; conserva testo, ID e split originali.
Il notebook non inventa etichette e non approva automaticamente le predizioni.

Carica in Colab il file **review_queue.json** dopo aver revisionato i testi.
Esegui installazione (sezione 1) e tutte le celle della sezione 9-bis:
la cella dedicata richiede il file, verifica etichette/split e mostra le disponibilita'.
Pending, esclusi ed esempi simulati non alimentano il training.
Sono necessari per ogni classe almeno 20 train, 5 validation e 5 test.
Il campione viene adattato automaticamente ai dati disponibili, entro il budget CPU.

Il precedente esperimento esterno resta disponibile da terminale con
`python train.py --data-source external --no-push`; non e' un ripiego automatico
quando la coda interna e' vuota.


## 9-bis. Retraining sul dataset interno approvato

1. Esegui l'installazione della sezione 1.
2. Esegui le celle qui sotto nell'ordine.
3. Carica **review_queue.json** quando richiesto. Deve contenere revisioni reali.
4. Controlla il riepilogo per classe/split e avvia la cella di training.
5. Leggi il confronto prima/dopo e il messaggio finale nel log del training, anche in caso di candidato rifiutato.

Il backbone resta congelato e si aggiorna solo la testa. La fonte nuova e'
il dataset interno Mastodon approvato; il replay proviene SOLO dal train TweetEval.
Gli split interni, assegnati alla raccolta, restano separati e persistenti.
La validation seleziona il checkpoint, mentre i test permettono il confronto finale.
Il budget massimo e' 1500 testi (1000 interni + 500 replay), ridotto automaticamente
quando la classe meno rappresentata contiene meno esempi. Massimo 3 epoche,
learning rate 1e-6 ed early stopping restano adatti a una prova contenuta anche su CPU.

Il log mostra sempre il confronto prima/dopo e, in caso di rifiuto, un avviso rosso.
Senza checkpoint ammissibile viene valutata l'ultima epoca solo a scopo diagnostico.
`--no-push` impedisce qualsiasi pubblicazione da questa prova Colab.
I report contengono i testi e le predizioni della prova e vengono salvati localmente.
Le soglie minime sono dimostrative: un campione piccolo non garantisce stime robuste.


In [ ]:
# Cartella separata per la copia dello script usata nella prova.
from pathlib import Path
import gc
import subprocess
import sys
import torch

RETRAIN_WORKDIR = Path.cwd() / "sentiment_retraining_colab"
RETRAIN_WORKDIR.mkdir(exist_ok=True)

# Se sono state eseguite le sezioni precedenti, spostiamo il modello
# di inferenza sulla CPU per lasciare memoria GPU al processo di training.
if "sentiment_pipeline" in globals():
    sentiment_pipeline.model.to("cpu")
    sentiment_pipeline.device = torch.device("cpu")
elif "model" in globals():
    model.to("cpu")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU disponibile:", torch.cuda.get_device_name(0))
else:
    print("GPU non disponibile: il training usera' la CPU.")
print("Cartella della prova:", RETRAIN_WORKDIR)


### Configurazione condivisa con lo script

Queste sono le costanti di training di `config.py`. La tolleranza e' un controllo di qualita': non va aumentata solo per far passare un candidato peggiore.


In [ ]:
# Copia delle costanti di training del progetto.
RETRAIN_CONFIG_SOURCE = '''
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}
RETRAIN_DATASET = "mteb/tweet_sentiment_extraction"
ORIGINAL_BENCHMARK_DATASET = "cardiffnlp/tweet_eval"
RETRAINED_MODEL_REPO_ID = "GiuliaC/sentiment-reputation-monitor-retrained"
REGRESSION_TOLERANCE = 0.02

'''
(RETRAIN_WORKDIR / "config.py").write_text(RETRAIN_CONFIG_SOURCE.lstrip("\n"), encoding="utf-8")


### Script di training condiviso

La cella seguente scrive su disco lo stesso `train.py` usato localmente e da GitHub Actions; non avvia ancora il training. `Trainer` seleziona automaticamente la GPU CUDA disponibile, altrimenti la CPU. La testa resta allenabile, mentre `requires_grad=False` impedisce di aggiornare i pesi del backbone.


In [ ]:
# Copia integrale di sentiment_reputation_mlops/train.py.
RETRAIN_SCRIPT_SOURCE = r'''
"""
Training/retraining su dataset interno Mastodon con etichette approvate.
Default: monitoring/review_queue.json, solo righe approved e revisione reale.
Il budget viene ridotto automaticamente se sono disponibili meno esempi.
La mancanza di dati approvati interrompe la prova prima di scaricare modelli.
--data-source external abilita esplicitamente il vecchio esperimento pubblico.

Training/retraining eseguibile da riga di comando, per il job "train" della
pipeline CI/CD (.github/workflows/train.yml, trigger manuale workflow_dispatch).

A differenza della valutazione principale del notebook (sezioni 1-9, sul test
set di cardiffnlp/tweet_eval), qui il fine-tuning avviene su un dataset
DIVERSO: mteb/tweet_sentiment_extraction (fonte: competizione Kaggle 2020, non
TweetEval). Il motivo: cardiffnlp/twitter-roberta-base-sentiment-latest e'
stato originariamente fine-tuned proprio su TweetEval per il task di sentiment
(vedi model card HuggingFace) - il dataset nuovo introduce esempi diversi; la quota di replay
riutilizza invece testi originali per preservare il compito precedente. Il nuovo dataset usa lo
stesso schema di etichette (0=negative, 1=neutral, 2=positive), quindi nessun
remapping aggiuntivo e' necessario.

Il training mescola dati nuovi e un campione del train TweetEval (replay):
1500 testi totali di default, di cui 1000 nuovi e 500 originali. Il replay
serve a contenere la perdita di prestazioni precedenti, senza garanzie.
Il backbone resta congelato; si aggiorna solo la testa. A ogni epoca, due
validation separate guidano early stopping e selezione del checkpoint in RAM.
Il checkpoint deve migliorare sui nuovi dati senza regredire oltre la tolleranza
sulla validation TweetEval. In assenza di checkpoint valido mostriamo comunque il test dell'ultima epoca
a scopo diagnostico, senza autorizzare la pubblicazione.

Il confronto prima/dopo viene fatto su due test set separati:
- il test set del dataset NUOVO (tweet_sentiment_extraction): misura se il
  fine-tuning aiuta davvero su dati mai visti dal modello di base;
- il test set di tweet_eval, lo stesso usato nella valutazione principale del
  notebook: e' il controllo di regressione, per verificare che il retraining
  non abbia peggiorato le prestazioni sul benchmark originale (catastrophic
  forgetting).

Il modello riaddestrato viene pubblicato su un repo HuggingFace dedicato
(RETRAINED_MODEL_REPO_ID) SOLO se supera il controllo di regressione: in
caso contrario lo script si interrompe con un errore esplicito e non
pubblica nulla. La promozione a "modello in produzione" (aggiornare
MODEL_NAME in config.py) resta comunque una decisione manuale, non
automatica.
"""
import argparse
import os
import json
from pathlib import Path
from uuid import uuid4

import pandas as pd

from datasets import Dataset as HFDataset, load_dataset
from huggingface_hub import HfApi
from huggingface_hub.utils import RepositoryNotFoundError
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    set_seed,
    TrainingArguments,
    pipeline,
)

from config import (
    LABEL_MAP,
    MODEL_NAME,
    ORIGINAL_BENCHMARK_DATASET,
    REGRESSION_TOLERANCE,
    RETRAIN_DATASET,
    RETRAINED_MODEL_REPO_ID,
)

LABEL2ID = {name: idx for idx, name in LABEL_MAP.items()}


def normalize_label(label: str) -> str:
    """
    Converte eventuali label tipo LABEL_0 nei nomi sentiment 
    """
    label = label.lower()
    if label.startswith("label_"):
        return LABEL_MAP[int(label.replace("label_", ""))]
    return label

def evaluate(model_pipeline, texts, true_labels, prediction_sink=None) -> dict:
    """
    Valuta `model_pipeline` su `texts`, confrontando le predizioni con
    `true_labels`.

    Restituisce accuracy e precision/recall/F1 macro - le stesse metriche
    della valutazione principale del notebook (sezione 9), cosi' i numeri
    prima/dopo il retraining (vedi main()) restano confrontabili con quelli.
    """
    predictions = [
        normalize_label(out["label"])
        # Una pipeline di HuggingFace, quando chiamata come una funzione su una lista di testi, 
        # accetta parametri extra che passa internamente.
        # pipe(KeyDataset(dataset, "text"), batch_size=8, truncation="only_first")
        # OUTPUT: # [{'label': 'POSITIVE', 'score': 0.9998743534088135}]
        for out in model_pipeline(texts, batch_size=16, truncation=True, max_length=128)
    ]
    if prediction_sink is not None:
        prediction_sink.extend(predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predictions, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(true_labels, predictions),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

def print_comparison(title: str, before: dict, after: dict) -> None:
    print(f"\n{title}")
    for key in before:
        print(f"  {key}: {before[key]:.4f} -> {after[key]:.4f}")


def resolve_base_model() -> str:
    """
    Decide da quale modello ripartire per il retraining.

    Se RETRAINED_MODEL_REPO_ID esiste gia' su HuggingFace Hub (pubblicato da
    un'esecuzione precedente di questo script), si riparte da li' - i
    retraining si incatenano invece di ripartire sempre dal modello base
    originale. Altrimenti si usa MODEL_NAME. 
    """
    try:
        HfApi().model_info(RETRAINED_MODEL_REPO_ID)
    except RepositoryNotFoundError:
        print(f"Nessun modello riaddestrato trovato su {RETRAINED_MODEL_REPO_ID}: "
              f"riparto dal modello base {MODEL_NAME}.")
        return MODEL_NAME
    print(f"Trovato un modello gia' riaddestrato su {RETRAINED_MODEL_REPO_ID}: riparto da li'.")
    return RETRAINED_MODEL_REPO_ID


def sample_df(hf_split, n, seed):
    """
    Converte uno split HuggingFace (es. retrain_dataset["test"]) in un
    campione pandas pronto all'uso: 
    lo sottocampiona a n righe casuali e aggiunge la colonna leggibile 
    "sentiment" accanto a "label" (0/1/2).
    """
    df = hf_split.to_pandas()
    df = df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)
    df["sentiment"] = df["label"].map(LABEL_MAP)
    return df



def split_retraining_data(hf_split, n_val, seed):
    """
    Prepara i nuovi dati da utilizzare per il retraining del modello.

    La funzione:
    1. converte il dataset Hugging Face in un DataFrame Pandas;
    2. elimina gli esempi senza testo o etichetta;
    3. elimina i testi duplicati per evitare che lo stesso testo possa
       comparire sia nel training sia nella validation;
    4. divide i dati in Training Set e Validation Set mantenendo
       la stessa proporzione delle classi (split stratificato);
    5. aggiunge la colonna 'sentiment', trasformando le label numeriche
       nei corrispondenti nomi delle classi tramite LABEL_MAP.

    La Validation viene separata prima di qualsiasi successivo
    campionamento dei dati di training, in modo che non venga utilizzata
    per addestrare il modello.

    Il Test Set rimane completamente separato e verrà utilizzato
    solamente per la valutazione finale del modello.
    """
    df = hf_split.to_pandas().dropna(subset=["text", "label"])
    df = df.drop_duplicates(subset=["text"])
    train_df, val_df = train_test_split(
        df, test_size=n_val, random_state=seed, stratify=df["label"]
    )
    train_df, val_df = train_df.copy(), val_df.copy()
    for frame in (train_df, val_df):
        frame["sentiment"] = frame["label"].map(LABEL_MAP)
    return train_df, val_df



def build_replay_training(new_pool, original_pool, n_train, n_replay, seed, excluded_texts):
    """
    Costruisce il dataset che verrà utilizzato per il retraining del modello.

    Il dataset finale combina due fonti:
    1. DATI NUOVI: post revisionati e approvati
    2. DATI DI REPLAY: esempi provenienti dal Training Set originale di TweetEval.
       Vengono aggiunti per mantenere durante il retraining anche
       esempi appartenenti alla distribuzione originale.
    `n_train` indica il numero TOTALE di esempi desiderati.

    Entrambi i gruppi vengono campionati in modo il più possibile
    bilanciato tra le tre classi.
    Prima del campionamento vengono inoltre:
    - eliminati esempi senza testo o label;
    - eliminati testi duplicati;
    - esclusi i testi appartenenti a validation e test;
    - evitati duplicati tra dati nuovi e dati di replay.

    Alla fine i due gruppi vengono uniti, mescolati e viene restituito
    un DataFrame pronto per essere utilizzato nel retraining.
    """
    if n_train - n_replay < 3 or n_replay < 0 or (0 < n_replay < 3):
        raise ValueError("Servono almeno 3 testi nuovi e zero oppure almeno 3 testi replay.")

    def sample_balanced(pool, count, excluded):
        """
        Estrae da un dataset (`pool`) il numero richiesto di esempi,
        distribuendoli nel modo più uniforme possibile tra le classi.
        Prima del campionamento:
        - elimina righe senza testo o label;
        - elimina testi duplicati;
        - elimina i testi presenti nell'insieme `excluded`.
        Successivamente divide `count` tra le tre classi.
        Se `count` non è perfettamente divisibile per 3, gli esempi
        rimanenti vengono distribuiti alle prime classi.
        Se una classe non contiene abbastanza esempi per raggiungere
        la quantità richiesta, viene sollevato un ValueError.

        Restituisce il campione bilanciato con anche la colonna
        `sentiment`, ottenuta dalla label numerica tramite LABEL_MAP.
        """
        # -----------------------------------------------------
        # PULIZIA DEL DATASET
        # -----------------------------------------------------
        pool = pool.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"])
        # esclusione dal training di esempi appartenenti a validation/test.
        pool = pool.loc[~pool["text"].isin(excluded)].copy()

        # CAMPIONAMENTO BILANCIATO PER CLASSE
        parts = []
        for index, label in enumerate(sorted(LABEL_MAP)):
            # Calcola quanti esempi dobbiamo prendere
            # dalla classe corrente.
            required = count // len(LABEL_MAP) + (index < count % len(LABEL_MAP))
             # prendo da pool gli esmepi della classe in questione
            group = pool.loc[pool["label"] == label]
            if len(group) < required:
                raise ValueError(f"Classe {label}: solo {len(group)} testi disponibili, richiesti {required}.")
            parts.append(group.sample(n=required, random_state=seed))
        # Unisce i campioni delle tre classi.
        result = pd.concat(parts, ignore_index=True)
        # Converte la label numerica nel nome del sentiment.
        result["sentiment"] = result["label"].map(LABEL_MAP)
        return result

    # ---------------------------------------------------------
    # 2. CAMPIONAMENTO DEI DATI NUOVI
    # ---------------------------------------------------------
    new = sample_balanced(new_pool, n_train - n_replay, excluded_texts)
    # ---------------------------------------------------------
    # 3. CAMPIONAMENTO DEI DATI DI REPLAY
    # ---------------------------------------------------------
    original = sample_balanced(original_pool, n_replay, set(excluded_texts) | set(new["text"]))
    result = pd.concat([new, original], ignore_index=True)
    print(f"Training con replay: {len(new)} testi nuovi + {len(original)} testi TweetEval "
          f"= {len(result)} totali. Validation e test esclusi.")
    # ---------------------------------------------------------
    # 4. PREPARAZIONE DEL DATASET FINALE
    # ---------------------------------------------------------
    # Conserva solamente le colonne necessarie al training:
    # text | sentiment
    # e mescola casualmente tutte le righe.
    # non rimangono raggruppati.
    result = result[["text", "sentiment"]].sample(frac=1, random_state=seed).reset_index(drop=True)
    result.attrs["source_counts"] = {
        "Training nuovo selezionato": new["sentiment"].value_counts().to_dict(),
        "Replay selezionato": original["sentiment"].value_counts().to_dict(),
    }
    return result


def save_analysis_report(directory, metadata, distributions, datasets):
    """
    Salva predizioni appaiate e dati della prova anche quando viene rifiutata.
    """
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    payload = {"metadata": metadata, "distributions": distributions, "datasets": datasets}
    path = directory / "report.json"
    # Prima serializziamo: in caso di errore non scriviamo un report parziale.
    content = json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False)
    path.write_text(content, encoding="utf-8")
    print(f"Analisi salvata in: {path.resolve()}")
    return path


class ValidationCheckpoint(TrainerCallback):
    """
    Gestisce la selezione del checkpoint migliore durante il retraining
    e implementa una forma di Early Stopping.
    Alla fine di ogni epoca il modello viene valutato su due Validation Set:
    - Validation nuova:
      serve a verificare se il retraining sta migliorando le prestazioni
      sui nuovi dati.
    - Validation originale:
      serve a controllare che il retraining non peggiori eccessivamente
      le prestazioni sui dati originali.

    Un checkpoint viene quindi salvato solamente se soddisfa ENTRAMBE
    le condizioni:
    1. la F1 sulla validation originale non è peggiorata oltre
       REGRESSION_TOLERANCE rispetto al modello iniziale;
    2. la F1 sulla validation nuova è migliorata di almeno min_delta
       rispetto al miglior risultato ottenuto fino a quel momento.
    Poiché durante il retraining il resto di RoBERTa è congelato,
    vengono conservati in RAM solamente i parametri addestrabili
    della testa di classificazione, evitando di salvare l'intero modello.

    Se per un numero di epoche pari a 'patience' non viene trovato
    nessun nuovo checkpoint che soddisfi entrambe le condizioni,
    il training viene interrotto tramite Early Stopping.

    NOTA:
    min_delta rappresenta una soglia pratica utilizzata per stabilire
    se il miglioramento è sufficientemente grande da essere considerato,
    ma non costituisce un test di significatività statistica.
    """

    def __init__(self, model_pipeline, new_df, original_df,
                 baseline_new_f1, baseline_original_f1, patience=2, min_delta=0.001):
        self.pipe = model_pipeline
        self.new_df = new_df
        # Validation Set originale, utilizzato per controllare che
        # il modello non perda eccessivamente le conoscenze precedenti.
        self.original_df = original_df
        self.best_f1 = baseline_new_f1
        self.original_f1 = baseline_original_f1
        self.patience = patience
        # miglioramento minimo della F1 sulla validation nuova necessario
        # per considerare il nuovo risultato realmente migliore.
        self.min_delta = min_delta
        self.stale_epochs = 0
        self.best_epoch = None
        self.best_head = None

    def consider(self, new_f1, original_f1, model, epoch):
        allowed = self.original_f1 - original_f1 <= REGRESSION_TOLERANCE
        improved = new_f1 > self.best_f1 + self.min_delta
        if allowed and improved:
            self.best_f1 = new_f1
            self.best_epoch = epoch
            self.best_head = {
                name: param.detach().cpu().clone()
                for name, param in model.named_parameters() if param.requires_grad
            }
            self.stale_epochs = 0
        else:
            self.stale_epochs += 1
        return allowed and improved

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        new = evaluate(self.pipe, self.new_df["text"].tolist(),
                       self.new_df["sentiment"].tolist())
        original = evaluate(self.pipe, self.original_df["text"].tolist(),
                            self.original_df["sentiment"].tolist())
        selected = self.consider(new["f1_macro"], original["f1_macro"], model, state.epoch)
        print(f"\nValidation epoca {state.epoch:.0f}: "
              f"F1 nuovo={new['f1_macro']:.4f}, F1 originale={original['f1_macro']:.4f}; "
              f"checkpoint selezionato: {'si' if selected else 'no'}.")
        if self.stale_epochs >= self.patience:
            print(f"Early stopping: {self.patience} epoche senza miglioramenti ammissibili.")
            control.should_training_stop = True
        return control

    def restore(self, model):
        if self.best_head is None:
            return False
        # Gli altri pesi sono rimasti congelati: basta ripristinare la testa.
        model.load_state_dict(self.best_head, strict=False)
        print(f"Ripristinato checkpoint dell'epoca {self.best_epoch:.0f}.")
        return True

def main(n_train: int, n_eval: int, epochs: int, seed: int, no_push: bool = False,
         n_val: int = 200, learning_rate: float = 1e-6, patience: int = 2,
         n_replay: int = 500, report_dir=None, reviewed_data=None, data_source="reviewed") -> None:
    if n_train - n_replay < 3 or n_replay < 0 or (0 < n_replay < 3) or min(n_eval, n_val, epochs, patience) <= 0 or learning_rate <= 0:
        raise ValueError("Servono almeno 3 testi nuovi, replay zero o almeno 3; gli altri parametri positivi.")
    # Il dataset interno e' il percorso principale: nessun ripiego silenzioso
    # sul dataset pubblico se mancano revisioni. Controllo PRIMA dei download.
    if data_source == "reviewed":
        from review_data import QUEUE_PATH, training_budget
        reviewed_data = reviewed_data or QUEUE_PATH
        n_train, n_replay, counts = training_budget(reviewed_data, n_train, n_replay)
        print("Esempi approvati per split e classe:", counts)
        print(f"Budget effettivo: {n_train - n_replay} interni + {n_replay} replay = {n_train}.")
    elif data_source != "external" or reviewed_data:
        raise ValueError("Scegliere reviewed oppure external; --reviewed-data vale solo per reviewed.")
    set_seed(seed)
    # 1. MODELLO BASE E TOKENIZER (vedi resolve_base_model per la logica di scelta)
    base_model_name = resolve_base_model()
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    # 2. DATASET DI RETRAINING (dati NUOVI, mai visti dal modello base)
    new_dataset_name = RETRAIN_DATASET
    if reviewed_data:
        from review_data import approved_splits, readiness
        ready, counts = readiness(reviewed_data)
        if not ready:
            raise ValueError(f"Dataset revisionato insufficiente: {counts}")
        # Usiamo solo etichette approvate, con split persistenti: i testi
        # gia' valutati non migrano nel training a ogni nuova raccolta.
        retrain_dataset = {name: HFDataset.from_list(rows)
                           for name, rows in approved_splits(reviewed_data).items()}
        new_dataset_name = "Mastodon - dataset interno con revisione umana"
        retrain_train_df = retrain_dataset["train"].to_pandas()
        retrain_train_df["sentiment"] = retrain_train_df["label"].map(LABEL_MAP)
        retrain_val_df = sample_df(retrain_dataset["validation"], n_val, seed)
        if n_train - n_replay > len(retrain_train_df):
            raise ValueError("Richiesti piu' testi nuovi di quelli approvati nel train.")
    else:
        print(f"Carico dataset di retraining: {RETRAIN_DATASET} (diverso da TweetEval)...")
        retrain_dataset = load_dataset(RETRAIN_DATASET)
        retrain_train_df, retrain_val_df = split_retraining_data(
            retrain_dataset["train"], n_val, seed
        )
    print("Fonte dei nuovi dati:", new_dataset_name)
    # Split "test" dello stesso dataset. > pipeline prende DataFrame pandas
    retrain_test_df = sample_df(retrain_dataset["test"], n_eval, seed)

    # ------------------------------------------------------------
    # 3. DATASET ORIGINALE (benchmark su cui il modello e' gia' stato addestrato)
    # ------------------------------------------------------------
    # Gli split validation/test servono come controllo di regressione: dopo il fine-tuning sui dati
    # nuovi, il modello deve continuare ad andare bene anche qui, altrimenti
    # ha "dimenticato" quello che sapeva gia' (catastrophic forgetting) - ed
    # e' proprio questo confronto a decidere se il modello va pubblicato o
    # scartato (vedi in fondo alla funzione). > pipeline prende DataFrame pandas
    print(f"Carico un campione di {ORIGINAL_BENCHMARK_DATASET} (controllo di regressione sul benchmark originale)...")
    original_dataset = load_dataset(ORIGINAL_BENCHMARK_DATASET, "sentiment")
    original_test_df = sample_df(original_dataset["test"], n_eval, seed)
    original_val_df = sample_df(original_dataset["validation"], n_val, seed)
    print(f"Validation separata: {len(retrain_val_df)} testi nuovi, "
          f"{len(original_val_df)} testi TweetEval.")

    # 4. Replay: parte dei testi proviene dal TRAIN originale per ricordare
    # il compito precedente senza aumentare il numero totale di esempi.
    # Escludiamo anche eventuali testi identici presenti negli split riservati.
    excluded_texts = set(retrain_val_df["text"])
    for held_out in (retrain_dataset["test"], original_dataset["validation"], original_dataset["test"]):
        excluded_texts.update(held_out["text"])
    if "validation" in retrain_dataset:
        excluded_texts.update(retrain_dataset["validation"]["text"])
    retraining_df = build_replay_training(
        retrain_train_df, original_dataset["train"].to_pandas(),
        n_train, n_replay, seed, excluded_texts,
    )

    # ------------------------------------------------------------
    # 5. VALUTAZIONE "PRIMA" DEL RETRAINING
    # ------------------------------------------------------------
    base_pipeline = pipeline(
        "sentiment-analysis",
        model=base_model_name,
        tokenizer=tokenizer,
        truncation=True,
        max_length=128,
    )

    predictions_before_new, predictions_before_original = [], []
    before_new = evaluate(
        base_pipeline, 
        retrain_test_df["text"].tolist(), 
        retrain_test_df["sentiment"].tolist(), prediction_sink=predictions_before_new
    )

    before_original = evaluate(
        base_pipeline, 
        original_test_df["text"].tolist(), 
        original_test_df["sentiment"].tolist(), prediction_sink=predictions_before_original
    )
    print("Metriche PRIMA del retraining, su dati nuovi:", before_new)
    print("Metriche PRIMA del retraining, su benchmark originale:", before_original)

    validation_new = evaluate(base_pipeline, retrain_val_df["text"].tolist(),
                              retrain_val_df["sentiment"].tolist())
    validation_original = evaluate(base_pipeline, original_val_df["text"].tolist(),
                                   original_val_df["sentiment"].tolist())
    print("F1 validation iniziale:", validation_new["f1_macro"],
          "(nuovo),", validation_original["f1_macro"], "(TweetEval)")

    # ------------------------------------------------------------
    # 6. PREPARAZIONE DEL DATASET PER IL TRAINER
    # ------------------------------------------------------------
    # Trainer di transformers lavora su un datasets.Dataset (non un DataFrame
    # pandas), con: label numerica (non stringa), testo gia' tokenizzato, e
    # nessuna colonna superflua (altrimenti il collator andrebbe in errore).
    # 1. Conversione in datasets.Dataset
    hf_train_dataset = HFDataset.from_pandas(retraining_df)
    # 2. Conversione in label numerica nella colonna "label"
    hf_train_dataset = hf_train_dataset.map(
        lambda batch: {"label": [LABEL2ID[s] for s in batch["sentiment"]]}, batched=True
    )
    # 3. tokenizzazione con il tokenizer
    # tokenizer(testo, ...) chiama tokenizer.__call__(), il modo standard per tokenizzare
    # OUTPUT = dizionario con input_ids/attention_mask
    hf_train_dataset = hf_train_dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=128), batched=True
    )
    # 4. Remove_columns per tenere solo le tre colonne che Trainer si aspetta.
    hf_train_dataset = hf_train_dataset.remove_columns(
        [c for c in hf_train_dataset.column_names if c not in ("input_ids", "attention_mask", "label")]
    )
    print(hf_train_dataset)

    # ------------------------------------------------------------
    # 7. FINE-TUNING 
    # ------------------------------------------------------------
    # Riutilizziamo il modello appena valutato: una sola copia in RAM/VRAM.
    # Le metriche PRIMA sono gia' state calcolate e conservate.
    model = base_pipeline.model
    # RoBERTa ha gia' appreso rappresentazioni utili dei tweet: manteniamo
    # fissi i suoi pesi e aggiorniamo solo la testa che decide il sentiment.
    # Questo riduce il lavoro di training, ma non garantisce un miglioramento:
    # anche la nuova testa deve superare il controllo di regressione finale.
    for parameter in model.base_model.parameters():
        parameter.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Backbone congelato: {trainable:,} parametri allenabili su {total:,} (solo testa).")
    # config dei parametri di training
    training_args = TrainingArguments(
        output_dir="./retraining_output",
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        learning_rate=learning_rate,
        seed=seed,
        dataloader_pin_memory=False,  # adatto anche al training su CPU
        logging_steps=10,
        save_strategy="no",  # il callback conserva in RAM solo la testa migliore
        report_to=[],  # niente integrazioni di logging esterne (es. wandb)
    )
    # Trainer utilizza le impostazioni di training_args per eseguire e gestire 
    # il ciclo di training, occupandosi automaticamente di operazioni come 
    # forward pass, calcolo della loss, backward pass, aggiornamento dei pesi,
    # gestione del device e passaggio tra modalità train() ed eval().
    checkpoint = ValidationCheckpoint(
        base_pipeline, retrain_val_df, original_val_df,
        validation_new["f1_macro"], validation_original["f1_macro"],
        patience=patience,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=hf_train_dataset,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        callbacks=[checkpoint],
    )
    print(f"\nRetraining su {len(hf_train_dataset)} esempi, {epochs} epoca/e...")
    trainer.train()
    validation_passed = checkpoint.restore(model)
    if not validation_passed:
        # Non interrompiamo qui: mostriamo comunque le metriche dell'ultima epoca.
        # Questo confronto diagnostico non rende pubblicabile il candidato.
        print("\033[31mATTENZIONE: nessun checkpoint ha superato la validation. "
              "Mostro il confronto dell'ULTIMA EPOCA solo a scopo diagnostico. "
              "Il candidato resta rifiutato.\033[0m", flush=True)
    else:
        print(f"Confronto del checkpoint selezionato: epoca {checkpoint.best_epoch:.0f}.")

    # ------------------------------------------------------------
    # 8. VALUTAZIONE "DOPO" IL RETRAINING E CONFRONTO
    # ------------------------------------------------------------
    # Stessa identica valutazione del punto 5, ma con la pipeline costruita
    # sul modello appena riaddestrato: il confronto prima/dopo e' quindi
    # sugli stessi identici testi di test, cambia solo il modello.
    retrained_pipeline = pipeline(
        "sentiment-analysis", 
        model=model, 
        tokenizer=tokenizer, 
        truncation=True, 
        max_length=128
    )
    predictions_after_new, predictions_after_original = [], []
    after_new = evaluate(
        retrained_pipeline, 
        retrain_test_df["text"].tolist(), 
        retrain_test_df["sentiment"].tolist(), prediction_sink=predictions_after_new)
    
    after_original = evaluate(
        retrained_pipeline,
        original_test_df["text"].tolist(), 
        original_test_df["sentiment"].tolist(), prediction_sink=predictions_after_original
    )

    print_comparison(f"Confronto su {new_dataset_name} (test separato dal retraining):", before_new, after_new)
    print_comparison(
        f"Confronto su {ORIGINAL_BENCHMARK_DATASET} (benchmark originale, controllo di regressione):",
        before_original,
        after_original,
    )

    # ------------------------------------------------------------
    # 9. GATE: il modello viene salvato solo se non e' peggiorato sul benchmark
    # ------------------------------------------------------------
    # Il retraining puo' anche "funzionare" (girare senza errori) e produrre
    # comunque un modello peggiore di quello in uso: qui e' il punto in cui
    # lo decidiamo, invece di limitarci a stampare i numeri e lasciare che
    # sia una persona a doverli leggere per accorgersene.
    f1_drop = float(before_original["f1_macro"] - after_original["f1_macro"])
    # Report persistente: il notebook puo' analizzare anche una prova rifiutata.
    # Ogni avvio del notebook usa una cartella diversa, evitando risultati obsoleti.
    distributions = dict(retraining_df.attrs["source_counts"])
    for name, frame in {
        "Training totale": retraining_df,
        "Validation nuovo": retrain_val_df, "Validation TweetEval": original_val_df,
        "Test nuovo": retrain_test_df, "Test TweetEval": original_test_df,
        "Pool train nuovo": retrain_train_df,
    }.items():
        distributions[name] = frame["sentiment"].value_counts().to_dict()
    distributions["Pool train TweetEval"] = (
        original_dataset["train"].to_pandas()["label"].map(LABEL_MAP).value_counts().to_dict()
    )
    diagnostic = {}
    for name, frame, before, after, preds_before, preds_after in [
        (new_dataset_name, retrain_test_df, before_new, after_new, predictions_before_new, predictions_after_new),
        (ORIGINAL_BENCHMARK_DATASET, original_test_df, before_original, after_original,
         predictions_before_original, predictions_after_original),
    ]:
        diagnostic[name] = {
            "metrics_before": before, "metrics_after": after,
            "rows": [
                {"text": text, "true": label, "before": first, "after": last}
                for text, label, first, last in zip(
                    frame["text"].tolist(), frame["sentiment"].tolist(), preds_before, preds_after)
            ],
        }
    save_analysis_report(
        report_dir or Path("retraining_output") / ("analysis_" + uuid4().hex[:12]),
        {"base_model": base_model_name, "new_dataset": new_dataset_name, "n_train": len(retraining_df), "n_replay": n_replay,
         "n_eval": n_eval, "n_val": n_val, "seed": seed, "learning_rate": learning_rate,
         "max_epochs": epochs, "completed_epochs": trainer.state.epoch,
         "evaluated_epoch": checkpoint.best_epoch if validation_passed else trainer.state.epoch,
         "validation_passed": validation_passed,
         "test_gate_passed": f1_drop <= REGRESSION_TOLERANCE,
         "candidate_accepted": validation_passed and f1_drop <= REGRESSION_TOLERANCE,
         "comparison": "checkpoint selezionato" if validation_passed else "ultima epoca diagnostica"},
        distributions, diagnostic,
    )
    # Il rifiuto viene comunicato DOPO aver stampato entrambi i confronti.
    # La validation resta vincolante anche se il test diagnostico e' buono.
    if not validation_passed:
        raise SystemExit(
            "\033[31mRetraining rifiutato sulla validation: nessuna epoca migliora "
            "la F1 nuova di oltre 0.001 rispettando la tolleranza sul benchmark. "
            "Il confronto sopra riguarda l'ultima epoca, non un checkpoint approvato. "
            "Il modello NON viene pubblicato.\033[0m"
        )
    if f1_drop > REGRESSION_TOLERANCE:
        raise SystemExit(
            f"\033[31mRetraining rifiutato: F1 macro sul benchmark originale sceso di "
            f"{f1_drop:.4f} (tolleranza {REGRESSION_TOLERANCE:.4f}). "
            "Il modello riaddestrato NON viene pubblicato.\033[0m"
        )

    if no_push:
        print(f"Controllo di regressione superato (calo F1 macro: {f1_drop:.4f}). "
              "Prova conclusa: --no-push esclude la pubblicazione su Hugging Face.")
        return

    print(
        f"\nControllo di regressione superato (calo F1 macro: {f1_drop:.4f}, "
        f"tolleranza {REGRESSION_TOLERANCE:.4f}). Pubblico il modello candidato su "
        f"{RETRAINED_MODEL_REPO_ID}..."
    )
    token = os.environ["HF_TOKEN"]
    model.push_to_hub(RETRAINED_MODEL_REPO_ID, token=token)
    tokenizer.push_to_hub(RETRAINED_MODEL_REPO_ID, token=token)
    print(
        f"Modello candidato pubblicato: https://huggingface.co/{RETRAINED_MODEL_REPO_ID}\n"
        "Per usarlo davvero in produzione, aggiorna MODEL_NAME in config.py "
        f"a '{RETRAINED_MODEL_REPO_ID}' - questa e' una decisione manuale, "
        "non automatica."
    )


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--n-train", type=int, default=1500, help="Esempi totali di retraining, inclusa la quota replay; esclusa la validation.")
    parser.add_argument("--n-replay", type=int, default=500, help="Quota del totale presa dal train TweetEval; 0 disattiva il replay.")
    parser.add_argument("--n-eval", type=int, default=200, help="Esempi di test per ciascun confronto prima/dopo.")
    parser.add_argument("--epochs", type=int, default=3, help="Massimo di epoche; early stopping attivo.")
    parser.add_argument("--n-val", type=int, default=200, help="Esempi per ciascuna validation.")
    parser.add_argument("--learning-rate", type=float, default=1e-6)
    parser.add_argument("--patience", type=int, default=2, help="Epoche senza miglioramenti ammissibili prima dello stop.")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--data-source", choices=["reviewed", "external"], default="reviewed", help="Default: dataset interno approvato; external ripete il vecchio esperimento.")
    parser.add_argument("--reviewed-data", default=None, help="JSON della coda: usa solo esempi approvati e split persistenti.")
    parser.add_argument("--report-dir", default=None, help="Cartella per il report di analisi; default: cartella univoca in retraining_output.")
    parser.add_argument("--no-push", action="store_true", help="Esegue training e valutazione senza pubblicare il modello.")
    args = parser.parse_args()
    main(n_train=args.n_train, n_eval=args.n_eval, epochs=args.epochs, seed=args.seed, no_push=args.no_push,
         n_val=args.n_val, learning_rate=args.learning_rate, patience=args.patience, n_replay=args.n_replay, report_dir=args.report_dir, reviewed_data=args.reviewed_data, data_source=args.data_source)
'''
(RETRAIN_WORKDIR / "train.py").write_text(RETRAIN_SCRIPT_SOURCE.lstrip("\n"), encoding="utf-8")
# Validazione della coda e budget condivisi con GitHub Actions.
REVIEW_DATA_SOURCE = r'''
"""
Coda di revisione umana condivisa da monitor.py, train.py e human_retrain.py.

Il flusso: monitor.py accoda i post con confidence bassa in 
monitoring/review_queue.json; una persona apre quel file e scrive a mano 
validated_label/review_status/reviewer/reviewed_at per ogni riga da 
approvare o escludere - questo modulo non assegna mai un'etichetta da solo. 
Le funzioni qui sotto leggono quella coda, verificano che le approvazioni
siano complete e coerenti, e preparano i tre split (train/validation/test)
che train.py userà per il retraining.

Per velocizzare l'approvazione: dopo aver scritto a mano solo
validated_label su una riga "pending", si può lanciare approve_reviewed.py,
che completa da solo review_status/reviewer/reviewed_at - senza decidere
mai un'etichetta al posto della persona.
"""
import hashlib
import json
import math
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

LABELS = ("negative", "neutral", "positive")
QUEUE_PATH = Path(__file__).parent / "monitoring" / "review_queue.json"
# Numero minimo di esempi approvati per OGNI classe in ciascuno split.
# Sono soglie per la dimostrazione, non una garanzia di accuratezza delle metriche.
MIN_COUNTS = {"train": 20, "validation": 5, "test": 5}


def text_key(text):
    """
    Crea una chiave identificativa stabile a partire dal contenuto di un testo.

    Prima normalizza il testo:
    - ignora le differenze tra maiuscole e minuscole;
    - elimina gli spazi multipli.

    Per esempio:
        "Hello  World"
        "hello world"
        "HELLO     WORLD"
    vengono tutti trasformati nella stessa forma:
        "hello world"

    Sul testo normalizzato viene poi calcolato un hash SHA-256,
    che funziona come una "impronta digitale" del contenuto.
    Lo stesso testo normalizzato produce quindi sempre la stessa chiave.
    Questa chiave viene utilizzata successivamente dal programma per:
    - riconoscere testi duplicati;
    - prendere decisioni riproducibili sul campionamento;
    - assegnare in modo stabile i testi agli split
      train / validation / test.

    IMPORTANTE:
    questa funzione crea solamente la chiave.
    Le decisioni sul campionamento e sugli split vengono effettuate
    successivamente utilizzando questa chiave.
    """
    return hashlib.sha256(" ".join(text.casefold().split()).encode("utf-8")).hexdigest()


def load_queue(path=QUEUE_PATH):
    """
    Legge la coda così com'è nel JSON, senza validare nulla.

    File assente = lista vuota (prima raccolta, non c'è ancora niente da
    leggere). 
    File presente ma JSON non valido = errore esplicito: non viene
    trattato come vuoto, per non perdere in silenzio revisioni già fatte se
    il file risulta corrotto o scritto solo a metà.

    Restituisce righe pending, approved ed excluded tutte mescolate insieme:
    è approved_splits (più sotto) a filtrarle e validarle prima del training.
    """
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else []


def save_queue(rows, path=QUEUE_PATH):
    """
    Salva su file lo stato completo della coda di revisione.
    Parametri:
    - rows: contiene tutte le righe che devono essere presenti nella coda;
    - path: indica il file in cui salvare la coda.

    La funzione:
    1. individua il percorso in cui deve essere salvato il file;
    2. crea automaticamente la cartella di destinazione se non esiste;
    3. converte l'intera coda `rows` in formato JSON;
    4. salva prima i dati in un file temporaneo;
    5. solo quando la scrittura è terminata, sostituisce il vecchio
       file della coda con quello appena creato.

    L'utilizzo di un file temporaneo rende il salvataggio più sicuro:
    se il programma si interrompesse mentre sta scrivendo i dati,
    il file originale della coda rimarrebbe intatto invece di rischiare
    di essere salvato solo parzialmente o corrotto.

    IMPORTANTE:
    questa funzione salva sempre l'INTERA coda ricevuta in `rows`.
    Inoltre si occupa solamente del salvataggio dei dati:
    non modifica lo stato delle righe..
    """
    # Converte il percorso in un oggetto Path per facilitarne la gestione.
    path = Path(path)
    # Crea la cartella, se non esiste già.
    path.parent.mkdir(parents=True, exist_ok=True)
    # Crea il percorso di un file temporaneo.
    temp = path.with_suffix(".tmp")
    # Converte tutta la coda in JSON e la salva nel file temporaneo.
    temp.write_text(json.dumps(rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    # Sostituzione del file al termine della scrittura
    temp.replace(path)


def enqueue(posts, predictions, model_name, scope, threshold=0.75, audit_rate=0.10, path=QUEUE_PATH):
    """
    Aggiunge alla coda i post che meritano una revisione umana.

    La funzione riceve:
    - `posts`: i post appena raccolti;
    - `predictions`: le predizioni del modello sugli stessi post;
    - `model_name`: nome del modello che ha prodotto le predizioni;
    - `scope`: contesto/ambito del monitoraggio;
    - `threshold`: soglia di default (0.75);
    - `audit_rate`: percentuale di predizioni ad alta confidence che
      vogliamo comunque controllare manualmente (default 10%);
    - `path`: file in cui è salvata la coda.

    I post possono entrare nella coda per DUE motivi:
    1. LOW CONFIDENCE
       Se confidence <= threshold, il modello non è abbastanza sicuro
       della propria predizione e il post viene mandato alla revisione.
    2. RANDOM AUDIT
       Anche alcune predizioni con confidence alta vengono selezionate
       come controllo. Con audit_rate=0.10 viene selezionato circa il 10%.
       La selezione non utilizza un numero casuale generato ogni volta:
       viene ricavata dalla chiave stabile prodotta da text_key().

    Prima di aggiungere un post, la funzione controlla inoltre che non
    sia già presente nella coda:
    - con lo stesso post_id;
    - oppure con lo stesso contenuto testuale normalizzato.

    Ogni nuovo post viene inserito con:
    - la predizione del modello e la relativa confidence;
    - il motivo per cui è stato selezionato;
    - review_status="pending";
    - nessuna validated_label, perché la vera etichetta dovrà essere
      assegnata successivamente da un revisore umano;
    - uno split stabile: train, validation oppure test.

    Lo split viene ricavato dalla chiave del testo con proporzioni
    approssimative:
        60% train
        20% validation
        20% test
    Alla fine viene salvata l'intera coda aggiornata.
    La funzione restituisce solamente il numero di NUOVI post aggiunti
    durante questa chiamata.
    """

    # I post e le predizioni devono corrispondere uno a uno.    
    # Esempio:
    # posts[0]       -> primo post
    # predictions[0] -> predizione del primo post   
    # Se le due liste hanno lunghezze diverse non possiamo sapere
    # correttamente quale predizione appartiene a quale post.
    if len(posts) != len(predictions):
        raise ValueError("Post e predizioni devono essere allineati.")
    rows = load_queue(path)
    known_ids = {row["post_id"] for row in rows}
    known_texts = {text_key(row["text"]) for row in rows}

    added = 0
    for post, prediction in zip(posts, predictions):
        # Crea l'impronta stabile del testo.
        key = text_key(post["text"])
        confidence = float(prediction["confidence"])
        label = prediction["sentiment"].lower()
        if label not in LABELS or not math.isfinite(confidence) or not 0 <= confidence <= 1:
            raise ValueError("Predizione non valida.")
        # ---------------------------------------------------------
        # MOTIVO 1: LOW CONFIDENCE
        # ---------------------------------------------------------
        # Se la confidence è minore o uguale alla soglia,
        # la predizione viene considerata poco sicura e quindi
        # deve essere controllata da una persona.
        uncertain = confidence <= threshold

        # ---------------------------------------------------------
        # MOTIVO 2: RANDOM AUDIT
        # ---------------------------------------------------------
        # VogliO controllare manualmente anche una piccola parte
        # delle predizioni che il modello considera sicure.
        # PrendO i primi 8 caratteri della chiave SHA-256 e li
        # trasformO in un numero compreso circa tra 0 e 1.
        # Se questo numero è inferiore ad audit_rate, il post viene
        # selezionato per il controllo.
        #
        # Con audit_rate=0.10 viene selezionato circa il 10% dei testi.
        sampled = int(key[:8], 16) / 2**32 < audit_rate

        if not (uncertain or sampled) or key in known_texts or post["post_id"] in known_ids:
            continue
        rows.append({
            **post, "predicted_label": label, "confidence": confidence,
            "model_name": model_name, "scope": scope,
            "selection_reason": "low_confidence" if uncertain else "random_audit",
            "collected_at": datetime.now(timezone.utc).isoformat(),
            "validated_label": None, "review_status": "pending",
            "reviewer": None, "reviewed_at": None, "review_is_simulated": False,
            # Un altro segmento della chiave assegna 6 valori su 10 al train,
            # 2 alla validation e 2 al test. Le proporzioni sono approssimative.
            # La correzione dell'etichetta non sposta il testo tra gli split.
            "split": "train" if int(key[8:16], 16) % 10 < 6 else (
                "validation" if int(key[8:16], 16) % 10 < 8 else "test"),
        })
        known_texts.add(key)
        known_ids.add(post["post_id"])
        added += 1
    save_queue(rows, path)
    return added


def approved_splits(path=QUEUE_PATH):
    """
    Prepara i dati approvati dalla revisione umana per il futuro retraining.

    La funzione legge tutti i post presenti nella coda di revisione,
    ma considera solamente quelli con:
        review_status = "approved"
    I post ancora "pending" oppure "excluded" vengono ignorati.

    Prima di utilizzare un post approvato, controlla che sia completo
    e valido. In particolare verifica che:
    - la revisione non sia simulata (`review_is_simulated=False`);
    - `validated_label` contenga una delle classi previste;
    - il post appartenga a uno split valido: train, validation o test;
    - siano presenti il nome del revisore e la data della revisione;
    - il testo non sia vuoto;
    - lo stesso testo non compaia più volte tra i dati approvati.

    Se anche un solo post approvato non supera questi controlli,
    viene sollevato un ValueError: la funzione non restituisce quindi
    un dataset parziale contenente solo gli esempi validi.

    I post validi vengono organizzati nei tre split:
        train
        validation
        test

    Per ogni post vengono conservati solamente:
    - il testo;
    - la label numerica corrispondente alla validated_label.

    Ad esempio, se LABELS è:
        ["negative" > 0, "neutral" > 1, "positive" > 2]

    La funzione restituisce quindi una struttura del tipo:
    {
        "train": [...],
        "validation": [...],
        "test": [...]
    }
    contenente solamente dati approvati e pronti per essere
    utilizzati nelle fasi successive del retraining.
    """

    # Prepara un dizionario con una lista vuota per ogni split.
    # MIN_COUNTS ad esempio contiene:
    # {"train": 20, "validation": 5, "test": 5}
    result = {split: [] for split in MIN_COUNTS}
    seen = set()
    for row in load_queue(path):
        if row.get("review_status") != "approved":
            continue
        if row.get("review_is_simulated") is not False:
            raise ValueError("Gli esempi approvati devono avere revisione reale esplicita.")
        if (row.get("validated_label") not in LABELS or row.get("split") not in result
                or not row.get("reviewer") or not row.get("reviewed_at")
                or not row.get("text", "").strip()):
            raise ValueError("Esempio approvato incompleto o non valido.")

        # Creo chiave
        key = text_key(row["text"])
        # se la chiave è in seen allora esiste un duplicato
        if key in seen:
            raise ValueError("Testo duplicato tra gli esempi approvati.")
        seen.add(key)
        # validated_label contiene una stringa: "negative" / "neutral" / "positive"
        # Per il training vogliamo invece la label numerica.
        # uso LABELS.index(...) per la conversione.
        result[row["split"]].append({"text": row["text"],
                                     "label": LABELS.index(row["validated_label"])})
    return result


def readiness(path=QUEUE_PATH):
    """
    Controlla se sono stati raccolti abbastanza esempi approvati
    per poter avviare il retraining.

    Prima di tutto usa approved_splits() per recuperare i dati approvati.
    Successivamente conta quanti esempi di ogni classe sono disponibili
    in ciascuno split. (MIN_COUNTS argginto per ogni split)
    Per poter considerare i dati "pronti", OGNI classe deve raggiungere
    il numero minimo richiesto dal proprio split.

    La funzione restituisce due informazioni:
    1. `ready`
       True se TUTTE le classi di TUTTI gli split raggiungono
       il numero minimo richiesto, False altrimenti.
    2. i conteggi effettivi per ogni split e per ogni classe,
       utili per capire quali dati mancano ancora.
    """
    splits = approved_splits(path)
    counts = {name: Counter(r["label"] for r in rows) for name, rows in splits.items()}
    ready = all(counts[split][label] >= minimum
                for split, minimum in MIN_COUNTS.items() for label in range(3))
    return ready, {split: {LABELS[i]: counts[split][i] for i in range(3)} for split in counts}


def dataset_fingerprint(path=QUEUE_PATH):
    """
    Crea una "firma" del dataset attualmente approvato per il retraining.
    La firma è un hash SHA-256 calcolato utilizzando solamente i dati
    che entrerebbero realmente nel training:
    - testo;
    - etichetta validata;
    - split (train / validation / test).

    Lo scopo è capire se il dataset è cambiato dall'ultimo tentativo
    di retraining.
    Se la firma attuale è uguale a quella salvata in precedenza,
    significa che i dati utilizzabili per il training sono gli stessi
    e quindi non è necessario ripetere lo stesso retraining.
    """
    splits = approved_splits(path)
    for rows in splits.values():
        rows.sort(key=lambda row: text_key(row["text"]))
    return hashlib.sha256(json.dumps(splits, sort_keys=True).encode()).hexdigest()


def training_budget(path=QUEUE_PATH, max_total=1500, max_replay=500):
    """
    Calcola quanti esempi utilizzare nel prossimo retraining.
    Il retraining utilizza due gruppi di dati:
    1. DATI NUOVI
       Post revisionati e approvati dagli utenti/revisori.
    2. DATI DI REPLAY
       Esempi provenienti da TweetEval, utilizzati insieme ai nuovi dati.
    Con i valori predefiniti l'obiettivo massimo è:
        1000 nuovi + 500 replay = 1500 esempi totali
    Tuttavia, la funzione adatta automaticamente questo budget alla
    quantità di dati nuovi realmente disponibili nel train.
    Per mantenere un dataset bilanciato tra le tre classi, guarda
    quanti esempi sono disponibili nella classe meno numerosa, 
    adattando a quel numero anche le altre classi.
    Esempio:
        negative = 100 |     neutral  = 80 |     positive = 20
    Viene quindi costruito un insieme bilanciato contenente al massimo:
        negative = 20  |     neutral  = 20 |     positive = 20

    Prima di calcolare il budget, la funzione controlla inoltre che:
    - i parametri max_total e max_replay siano validi;
    - il file contenente i dati revisionati esista;
    - readiness() confermi che siano già disponibili almeno
      i dati minimi richiesti per ogni classe e ogni split.

    IMPORTANTE:
    questa funzione calcola solamente quanti esempi nuovi e di replay
    train.py dovrà successivamente richiedere.

    Restituisce:
    - numero totale di esempi da utilizzare;
    - numero di esempi di replay;
    - conteggi degli esempi approvati per split e classe.
    """
    # 1. CONTROLLO DEL BUDGET RICHIESTO
    # max_total - max_replay rappresenta quanti dati NUOVI vogliamo utilizzare.
    if max_total - max_replay < 3 or max_replay < 0:
        raise ValueError("Budget non valido: servono almeno 3 testi nuovi.")
    # 2. CONTROLLO DELL'ESISTENZA DEI DATI
    if not Path(path).is_file():
        raise ValueError(f"Dataset interno assente: {path}. Raccogli e revisiona i post prima del training.")
    # 3. CONTROLLO DELLA QUANTITÀ MINIMA DI DATI
    ready, counts = readiness(path)
    if not ready:
        raise ValueError(f"Etichette approvate insufficienti. Minimi per classe: {MIN_COUNTS}. Disponibili: {counts}")
    # 4. BUDGET IDEALE DEI DATI NUOVI
    requested_new = max_total - max_replay
    # 5. ADATTA IL BUDGET AI DATI REALMENTE DISPONIBILI
    # new_count numero di esempi tot (min(requested_new) * 3 (train, test, split))
    new_count = min(requested_new, min(counts["train"].values()) * 3)
    # 6. CALCOLA LA QUOTA DI REPLAY
    # Se abbiamo dovuto ridurre i dati nuovi, riduciamo
    # proporzionalmente anche i dati di replay.
    # nuovi : replay =  1000 : 500 = cioè circa 2 : 1.
    replay_count = round(new_count * max_replay / requested_new)
    # Se il calcolo produce una quota di replay maggiore di zero
    # ma inferiore a 3, viene considerata troppo piccola.
    if 0 < replay_count < 3:
        raise ValueError("Quota replay troppo piccola: aumentare il budget oppure usare zero.")
    return new_count + replay_count, replay_count, counts
'''
(RETRAIN_WORKDIR / "review_data.py").write_text(REVIEW_DATA_SOURCE.lstrip("\n"), encoding="utf-8")


### Caricamento e controllo del dataset interno
Carica la coda revisionata scaricata dal repository. Se il controllo segnala dati insufficienti, raccogli e revisiona altri testi: il notebook non passa automaticamente al dataset pubblico.


In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
from IPython.display import display

# Su Colab: carica il JSON dal computer se non e' gia' presente nella sessione.
REVIEWED_DATA = (Path.cwd() / "review_queue.json").resolve()
if not REVIEWED_DATA.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Copia review_queue.json nella cartella del notebook.")
    files.upload()
if not REVIEWED_DATA.is_file():
    raise FileNotFoundError("Serve un file chiamato review_queue.json.")

# Carichiamo il validatore appena scritto dalla cella precedente,
# senza usare un'eventuale vecchia versione presente nella sessione Python.
spec = importlib.util.spec_from_file_location("review_helper", RETRAIN_WORKDIR / "review_data.py")
review_helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(review_helper)
ready, counts = review_helper.readiness(REVIEWED_DATA)
display(pd.DataFrame.from_dict(counts, orient="index"))
if not ready:
    raise ValueError("Revisioni insufficienti: servono per ogni classe almeno "
                     "20 train, 5 validation e 5 test. Completare la revisione prima di proseguire.")
print("Dataset approvato pronto:", REVIEWED_DATA)


In [ ]:
# Configurazione contenuta anche per CPU: piu' dati, massimo 3 epoche.
# Aumentare questi valori aumenta il lavoro, non garantisce un miglioramento.
# Limiti massimi, non quantita' da inventare se mancano esempi approvati.
#
# CONFIGURAZIONE SCELTA dopo uno sweep di prove (vedi la cella markdown
# subito sotto "Come interpretare il risultato" per il dettaglio dei
# tentativi fatti e perche' questa e' la combinazione giusta). Punti chiave:
# replay 1:1 (66 nuovi + 66 replay, non il 2:1 di default), LEARNING_RATE
# piu' alto del default (2e-6) con PATIENCE alta (10 epoche) per lasciare
# al training il tempo di trovare un checkpoint ammissibile, e N_EVAL/N_VAL
# alzati a 500 (dal default 200): con soli 200 esempi di benchmark il calo
# misurato tra validation e test poteva differire abbastanza da cambiare
# l'esito vicino alla soglia di tolleranza - con 500 il risultato e' stabile.
# Configurazione "demo" originale, da ripristinare se si vuole tornare al
# comportamento di default: MAX_TRAIN=1500, MAX_REPLAY=500, EPOCHS=3,
# LEARNING_RATE=1e-6, PATIENCE=2, N_EVAL=200, N_VAL=200.
MAX_TRAIN = 132
MAX_REPLAY = 66
N_EVAL = 500
N_VAL = 500
EPOCHS = 10
LEARNING_RATE = 2e-6
PATIENCE = 10
SEED = 42

# Usa il JSON controllato nella cella precedente. Mantiene il rapporto 2:1
# quando riduce il campione alla disponibilita' della classe meno numerosa.
N_TRAIN, N_REPLAY, approved_counts = review_helper.training_budget(
    REVIEWED_DATA, MAX_TRAIN, MAX_REPLAY)
print(f"Training effettivo: {N_TRAIN - N_REPLAY} testi interni + {N_REPLAY} replay.")

if N_TRAIN - N_REPLAY < 3 or N_REPLAY < 0 or (0 < N_REPLAY < 3) or min(N_EVAL, N_VAL, EPOCHS, PATIENCE) <= 0 or LEARNING_RATE <= 0:
    raise ValueError("Servono almeno 3 testi nuovi, replay zero o almeno 3, dimensioni, epoche, patience e learning rate positivi.")

from uuid import uuid4
# Cartella univoca: una prova fallita non riutilizza un vecchio report.
ANALYSIS_DIR = RETRAIN_WORKDIR / ("analysis_" + uuid4().hex[:12])
command = [
    sys.executable, "-u", "train.py",
    "--n-train", str(N_TRAIN),
    "--n-replay", str(N_REPLAY),
    "--n-eval", str(N_EVAL),
    "--epochs", str(EPOCHS),
    "--seed", str(SEED),
    "--n-val", str(N_VAL),
    "--learning-rate", str(LEARNING_RATE),
    "--patience", str(PATIENCE),
    "--no-push",
    "--report-dir", str(ANALYSIS_DIR),
]
command.extend(["--data-source", "reviewed", "--reviewed-data", str(REVIEWED_DATA)])
print("Comando:", " ".join(command))
# Il processo separato usa config.py dalla cartella della prova
# e libera i propri modelli dalla memoria quando termina.
# Colab non mostra sempre stdout/stderr ereditati da un sottoprocesso.
# Leggiamo entrambi i flussi e li stampiamo nella cella, riga per riga:
# cosi' sono visibili anche i confronti prima/dopo e gli eventuali errori.
training_log = []
with subprocess.Popen(
    command,
    cwd=RETRAIN_WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
) as process:
    try:
        for line in process.stdout:
            training_log.append(line)
            print(line, end="", flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        # Conserviamo anche il log su disco, per rileggerlo senza rifare il training.
        training_log_path = RETRAIN_WORKDIR / "training_colab.log"
        training_log_path.write_text("".join(training_log), encoding="utf-8")

print("\nLog salvato in:", training_log_path)
print("Codice di uscita:", return_code)
if return_code == 0:
    print("Prova completata e controllo superato. Nessuna pubblicazione (--no-push).")
else:
    print("Leggi il log precedente: 'Retraining rifiutato' indica un candidato "
          "peggiorativo; un traceback indica invece un errore di esecuzione.")


### Come interpretare il risultato

- **Tempo di training:** `train_runtime` esclude download e valutazioni.
- **Selezione:** il log mostra F1 sulle due validation a ogni epoca, early stopping ed epoca ripristinata. Il checkpoint della testa resta in RAM solo durante la prova.
- **Qualita':** confronta accuracy e F1 macro prima/dopo su entrambi i dataset.
- **Controllo superato:** il calo F1 sul benchmark resta entro la tolleranza; questo non implica necessariamente un miglioramento sui nuovi dati.
- **Retraining rifiutato:** il candidato non supera la validation oppure il controllo finale sul benchmark. Le metriche sono comunque mostrate. Se manca un checkpoint ammissibile, riguardano l'ultima epoca e sono accompagnate da un avviso rosso; la pubblicazione resta bloccata.
- Confrontando CPU e GPU, mantieni inizialmente gli stessi parametri. Differenze numeriche tra hardware sono possibili anche con seed fisso.

Compila le osservazioni con i numeri prodotti da questa esecuzione, senza riutilizzare quelli di una prova precedente.


### Come è stata scelta questa configurazione

I valori impostati sopra (`MAX_TRAIN=132`, `MAX_REPLAY=66`, `LEARNING_RATE=2e-6`, `EPOCHS=10`, `PATIENCE=10`, `N_EVAL=500`, `N_VAL=500`) non sono i default "demo" del progetto: sono il risultato di uno sweep di prove fatte a mano, tutte a testa congelata (backbone di RoBERTa non allenato), sullo stesso dataset interno via via arricchito con nuove revisioni umane. Le riportiamo qui perché "perché proprio questi numeri" è una domanda d'esame naturale.

**Primi tentativi (dataset piccolo, replay 2:1 di default)**: con `LEARNING_RATE` tra 1e-7 e 5e-5, ogni epoca in cui la F1 sui dati nuovi migliorava abbastanza da superare `min_delta`, il calo di F1 sul benchmark originale aveva già superato `REGRESSION_TOLERANCE` (0,02) — anche con 50 epoche di margine. Un `LEARNING_RATE` troppo basso (1e-7) non faceva invece muovere il modello a sufficienza: né miglioramento sui dati nuovi né calo apprezzabile sull'originale.

**Aumentare il replay** (fino a 1:2 nuovi:replay) riduceva il calo sul benchmark ma non risolveva da solo il problema: il miglioramento sui dati nuovi restava troppo lento o troppo "costoso" in termini di regressione.

**Il salto di qualità: più dati approvati.** Passando da un dataset con la classe più scarsa (`neutral`) a sole 32 righe nel train a uno con 66, il divario tra "quanto migliora il nuovo" e "quanto regredisce l'originale" si è ristretto sensibilmente — prova diretta che parte del problema era semplicemente insufficienza di segnale, non solo iperparametri sbagliati.

**Il dettaglio che ha davvero sbloccato il risultato**: con `N_EVAL`/`N_VAL` al default (200), un'epoca poteva superare il controllo sulla *validation* (calo quasi nullo, es. 0,0004) e venire comunque rifiutata dal controllo finale sul *test* (calo 0,0265, sopra soglia) — stesso identico checkpoint, due campioni diversi di `tweet_eval` da 200 esempi, due misure del calo abbastanza diverse da cambiare l'esito. Portando `N_EVAL`/`N_VAL` a 500, la misura è diventata stabile tra validation e test (calo coerente, circa 0,0015-0,007 in entrambi i casi): è a questo punto che un'epoca (la 6ª, con `LEARNING_RATE=2e-6`) ha superato *entrambi* i controlli.

**In sintesi, cosa ha contato davvero**, in ordine di impatto osservato:
1. più dati approvati (soprattutto sulla classe meno numerosa);
2. un campione di valutazione più grande (200 → 500), per non decidere il confine accetta/rifiuta sul rumore statistico di un campione piccolo;
3. un `LEARNING_RATE` intermedio (2e-6) e abbastanza `PATIENCE`/`EPOCHS` da lasciare al training il tempo di trovare il punto giusto, senza fermarsi troppo presto né continuare oltre fino a peggiorare di nuovo.

Non è stato invece necessario sbloccare il backbone (fine-tuning completo): un test di confronto in tal senso era stato preparato ma non è servito, proprio perché la combinazione sopra si è dimostrata sufficiente con la sola testa allenabile.


## 10. Pipeline CI/CD e monitoraggio continuo

Le Fasi 2 e 3 della consegna (pipeline automatizzata per training/test/deploy, sistema di monitoraggio continuo) non sono simulazioni dentro questo notebook: sono implementate come repository GitHub reale, con codice che gira davvero tramite GitHub Actions. Qui vengono solo descritte; il codice vive in [`sentiment_reputation_mlops/`](sentiment_reputation_mlops/) e nei workflow alla radice del repository (vedi struttura sotto).

In [ ]:
# ============================================================
# STRUTTURA DEL REPOSITORY (riferimento — i file vivono nel repo, non qui)
# ============================================================

REPO_STRUCTURE = """
<radice del repository GitHub>
├── .github/workflows/
│   ├── ci.yml                # job "test" (pytest) + job "deploy" (HuggingFace Space, dopo i test)
│   ├── train.yml             # job "train", trigger manuale (workflow_dispatch)
│   └── monitor.yml           # job "monitor", schedulato (cron) + trigger manuale
└── sentiment_reputation_mlops/
    ├── requirements.txt      # dipendenze del repository (transformers, gradio, requests, ecc.)
    ├── config.py             # costanti centralizzate: modello, dataset, soglie, repo HuggingFace
    ├── predictor.py          # SentimentPredictor: carica il modello una volta, espone predict()
    ├── app.py                # demo Gradio, usa SentimentPredictor da predictor.py
    ├── train.py              # retraining su dati mai visti dal modello base + gate di promozione
    ├── monitor.py            # monitoraggio del sentiment su post reali (Mastodon), con baseline storica
    ├── deploy_to_hf.py       # pubblica questa cartella come HuggingFace Space
    ├── README.md             # frontmatter richiesto da HuggingFace Space
    ├── conftest.py           # vuoto: serve solo perche' pytest trovi predictor.py da tests/
    ├── monitoring/history.json  # baseline storica, aggiornata automaticamente dal job "monitor"
    └── tests/
        ├── test_app.py     # test_model_loads, test_known_examples, casi limite, schema di output
        └── test_app.py       # verifica che app.py (Gradio) funzioni, non solo predictor.py
"""

print(REPO_STRUCTURE)


**Osservazione sulla pipeline CI/CD e sul monitoraggio**

**`config.py`** centralizza le costanti usate dagli altri script (nome del modello, dataset di retraining, repository HuggingFace di destinazione, soglie) — stesso principio della `Config` del notebook (sezione 2), ma per il codice che vive nella repository.

**Job `test`** (`ci.yml`): installa le dipendenze ed esegue `pytest` ad ogni push o pull request su `main` che tocchi `sentiment_reputation_mlops/`. `app.py` e `tests/test_app.py` importano entrambi `SentimentPredictor` da `predictor.py`, che accentra il caricamento del modello in un solo posto. Oltre allo smoke test, `test_app.py` verifica lo schema di output (etichetta valida, confidence in [0, 1]) su casi limite (testo vuoto, molto lungo, lingua diversa dall'inglese, emoji); `test_app.py` verifica che anche `app.py` — non solo `predictor.py` — funzioni davvero.

**Job `deploy`** (`ci.yml`, `needs: test`, solo su push a `main`): pubblica `app.py` come HuggingFace Space tramite `huggingface_hub`, creandolo al primo deploy se non esiste ancora.

**Job `train`** (`train.yml`, trigger manuale `workflow_dispatch`): esegue `train.py`, che riallena il modello su `mteb/tweet_sentiment_extraction` — un dataset **diverso** da quello di valutazione (`tweet_eval`), perche' il modello base e' gia' stato fine-tuned proprio su TweetEval (lo dice la sua model card su HuggingFace): riallenarlo sugli stessi dati non introdurrebbe nessuna informazione nuova. Il confronto prima/dopo viene fatto sia sul dataset nuovo sia su un campione di `tweet_eval` (controllo di regressione, per verificare che il modello non abbia "dimenticato" quello che sapeva gia' fare bene). Il modello riaddestrato viene pubblicato su un repository HuggingFace dedicato **solo se** il calo di F1 macro sul benchmark originale resta entro una soglia di tolleranza: altrimenti il job fallisce esplicitamente e non pubblica nulla. La promozione a "modello in produzione" resta comunque una decisione manuale.

**Job `monitor`** (`monitor.yml`, schedulato una volta al giorno + trigger manuale): esegue `monitor.py`, che scarica testi pubblici reali da un'istanza Mastodon (nessuna etichetta necessaria: il monitoraggio del drift di sentiment si basa solo sulle predizioni del modello, non sull'accuratezza), li classifica con il modello attuale, calcola la quota di sentiment negativo del batch e la confronta con una baseline storica (media + 1 deviazione standard delle esecuzioni precedenti). Il risultato viene salvato in `monitoring/history.json`, che il workflow ricommitta nel repository ad ogni esecuzione: e' cosi' che la baseline cresce davvero nel tempo, invece di essere ricalcolata da zero ogni volta.

In un progetto reale aggiungerei anche controlli sul formato dei dati e linting nella pipeline di test; il punto importante resta che il modello non dovrebbe arrivare in produzione solo perche' il notebook funziona: serve una catena automatica che controlli codice, dipendenze e comportamento minimo del sistema nel tempo.

## 11. Conclusioni finali

Il progetto usa un modello di sentiment analysis gia' pronto (CardiffNLP, addestrato su testi social) per classificare testi in `negative`/`neutral`/`positive`, e ne valuta criticamente le prestazioni su un benchmark pubblico (`tweet_eval`) prima di considerarlo adatto all'uso.

Le Fasi 2 e 3 della consegna — pipeline CI/CD automatizzata e sistema di monitoraggio continuo — non sono descritte solo a parole: sono implementate come repository GitHub reale (sezione 10), con codice che gira davvero tramite GitHub Actions:

- **training** automatizzato su un dataset diverso da quello di valutazione (per introdurre davvero informazione nuova al modello), con pubblicazione del modello condizionata a un controllo di regressione;
- **test di integrazione** su schema di output, casi limite, e sull'applicazione stessa (`app.py`), non solo sul modello;
- **deploy** automatico su HuggingFace Space dopo il superamento dei test;
- **monitoraggio continuo** del sentiment su dati social reali (Mastodon), con una baseline storica che si aggiorna da sola nel repository nel tempo.